# compare

Load the three DRL runs, validate their metadata, evaluate five schemes on the same test window, and export the comparison table plus the four requested figures.


In [ ]:
from pathlib import Path
import sys

project_root = Path.cwd().resolve()
while project_root != project_root.parent and not (project_root / "configs").exists():
    project_root = project_root.parent
if not (project_root / "configs").exists():
    raise RuntimeError("Could not locate the project root from the notebook working directory.")
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from IPython.display import display

import pandas as pd

from configs import compose_experiment_config
from scripts.utils.experiment_notebook_utils import resolve_madrl_model_root
from scripts.utils.grid_notebook_workflow import (
    apply_notebook_experiment_settings,
    collect_madrl_rollout,
    collect_mpc_rollout,
    compare_rollout_metrics,
    plot_net_load_comparison,
    plot_power_balance_comparison,
    plot_rollout_comparison_dashboard,
    plot_voltage_profile_comparison,
    validate_compare_model_bundles,
)
from scripts.utils.madrl_shared_data import ensure_madrl_shared_data
from scripts.utils.project_paths import project_root


In [ ]:
PROJECT_ROOT = project_root()
DATA_DIR = PROJECT_ROOT / "data"
CHECKPOINT_ROOT = None
PREDICTION_MODE = "normal"  # normal = LSTM forecast

EXPORT_SUBSIDY_EUR_PER_KWH = 0.079
LAMBDA_THROUGHPUT = 0.0
W_ACTION_PEN = 0.0
EVAL_W_VOLTAGE_PEN = 10.0
EVAL_W_LINE_PEN = 10.0
EVAL_W_TRAFO_PEN = 10.0

DRL_RUN_SPECS = {
    "MADRL + No Safety": {"algorithm": "MATD3", "experiment_name": "train_base", "model_root": None},
    "MADRL + Safety Penalty": {"algorithm": "MATD3", "experiment_name": "train_base_safe", "model_root": None},
    "MADRL + Safety Projection": {
        "algorithm": "MATD3_SAFE_POC",
        "experiment_name": "train_projection_safe",
        "model_root": None,
    },
}

EXPORT_DIR = PROJECT_ROOT / "artifacts" / "compare"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)


In [ ]:
resolved_model_roots = {}
for label, spec in DRL_RUN_SPECS.items():
    resolved_model_roots[label] = resolve_madrl_model_root(
        algorithm=spec["algorithm"],
        prediction_mode=PREDICTION_MODE,
        experiment_name=spec["experiment_name"],
        model_root=spec.get("model_root"),
        root=PROJECT_ROOT,
        checkpoint_root=CHECKPOINT_ROOT,
    )

bundles = validate_compare_model_bundles(resolved_model_roots)
display(
    pd.DataFrame(
        {
            "scheme": list(resolved_model_roots.keys()),
            "model_root": [str(path) for path in resolved_model_roots.values()],
        }
    )
)


In [ ]:
reference_bundle = bundles["MADRL + No Safety"]
reference_experiment = reference_bundle["experiment_controls"]
reference_data = reference_bundle["data_controls"]
reference_battery = reference_bundle["battery_controls"]
reference_train = reference_bundle["train_controls"]

forecast_controls = dict(reference_experiment.get("forecast_controls") or {})
if forecast_controls:
    forecast_controls["auto_train_missing"] = False

cfg = compose_experiment_config(
    profile=reference_train.get("profile", "base"),
    algorithm="MATD3",
    model_family=reference_train.get("model_family", "mlp"),
    data_dir=DATA_DIR,
    device=reference_experiment.get("device_request"),
    runtime_mode=reference_experiment.get("runtime_mode", "performance"),
    seed=int(reference_experiment.get("seed", 0)),
    require_cuda=reference_experiment.get("require_cuda"),
)
apply_notebook_experiment_settings(
    cfg,
    prediction_mode=reference_data["prediction_mode"],
    test_start_date=reference_data.get("test_start_date"),
    test_end_date=reference_data.get("test_end_date"),
    agent_profiles=reference_data["agent_profiles"],
    agent_bus_ids=reference_data.get("agent_bus_ids"),
    load_scale=reference_data.get("load_scale"),
    pv_scale=reference_data.get("pv_scale"),
    battery_controls=reference_battery,
    forecast_controls=forecast_controls,
    future_horizon=reference_data.get("future_horizon"),
    train_year=reference_data.get("train_year"),
    test_year=reference_data.get("test_year"),
)
cfg.reward.export_subsidy_eur_per_kwh = EXPORT_SUBSIDY_EUR_PER_KWH
cfg.reward.lambda_throughput = LAMBDA_THROUGHPUT
cfg.reward.w_action_pen = W_ACTION_PEN
cfg.reward.w_voltage_pen = EVAL_W_VOLTAGE_PEN
cfg.reward.w_line_pen = EVAL_W_LINE_PEN
cfg.reward.w_trafo_pen = EVAL_W_TRAFO_PEN

shared_data_result = ensure_madrl_shared_data(cfg)
cfg.runtime.shared_data_dir = str(shared_data_result.shared_data_dir)
cfg.runtime.shared_data_signature = str(shared_data_result.signature_hash)
display(
    {
        "test_start_date": reference_data.get("test_start_date"),
        "test_end_date": reference_data.get("test_end_date"),
        "shared_data_dir": cfg.runtime.shared_data_dir,
        "shared_data_signature": cfg.runtime.shared_data_signature,
        "shared_data_reused": bool(shared_data_result.reused),
    }
)


In [ ]:
mpc_perfect = collect_mpc_rollout(cfg, prediction_mode="perfect", label="MPC + Perfect Forecast")
mpc_lstm = collect_mpc_rollout(cfg, prediction_mode="normal", label="MPC + LSTM Forecast")
madrl_base = collect_madrl_rollout(
    cfg,
    model_root=resolved_model_roots["MADRL + No Safety"],
    algorithm="MATD3",
    experiment_name="train_base",
    checkpoint_root=CHECKPOINT_ROOT,
    label="MADRL + No Safety",
)
madrl_safe = collect_madrl_rollout(
    cfg,
    model_root=resolved_model_roots["MADRL + Safety Penalty"],
    algorithm="MATD3",
    experiment_name="train_base_safe",
    checkpoint_root=CHECKPOINT_ROOT,
    label="MADRL + Safety Penalty",
)
madrl_projection = collect_madrl_rollout(
    cfg,
    model_root=resolved_model_roots["MADRL + Safety Projection"],
    algorithm="MATD3_SAFE_POC",
    experiment_name="train_projection_safe",
    checkpoint_root=CHECKPOINT_ROOT,
    label="MADRL + Safety Projection",
)

rollouts = [mpc_perfect, mpc_lstm, madrl_base, madrl_safe, madrl_projection]
metrics_df = compare_rollout_metrics(*rollouts)
display(metrics_df)


In [ ]:
display(plot_rollout_comparison_dashboard(metrics_df))
display(plot_voltage_profile_comparison(*rollouts))
display(plot_net_load_comparison(*rollouts))
display(plot_power_balance_comparison(*rollouts))
